In [1]:
import os
import subprocess
import vsp
import numpy as np
import pandas as pd
from pyDOE2 import lhs

# 경로 세팅
TEMPLATE_VSP3 = "example_output.vsp3"  # 네 템플릿 파일
OUTPUT_DIR = "generated_vspaero_files"
VSPAERO_EXE_PATH = r"C:\Users\dsmsj\OpenVSP-3.42.3-win64\vspaero.exe"  # 네 환경 맞춤
os.makedirs(OUTPUT_DIR, exist_ok=True)

# DegenGeom 타입 정의 (버전에 따라 직접 세팅)
DEGEN_GEOM_VSPAERO_TYPE = 5

# 샘플 개수 및 범위
N_SAMPLES = 10
SPAN_RANGE = (15.0, 25.0)
AREA_RANGE = (30.0, 50.0)

# LHS 샘플링
np.random.seed(42)
lhs_samples = lhs(2, samples=N_SAMPLES)
spans = SPAN_RANGE[0] + (SPAN_RANGE[1] - SPAN_RANGE[0]) * lhs_samples[:, 0]
areas = AREA_RANGE[0] + (AREA_RANGE[1] - AREA_RANGE[0]) * lhs_samples[:, 1]

# 파일 생성 및 해석 루프
for idx, (span, area) in enumerate(zip(spans, areas)):
    print(f"\n🛠️ 샘플 {idx} 처리 중...")
    try:
        # 모델 불러오기
        vsp.ClearVSPModel()
        vsp.ReadVSPFile(TEMPLATE_VSP3)

        # 파라미터 수정
        wing_id = vsp.FindGeom("WingGeom", 0)
        vsp.SetParmVal(wing_id, "TotalSpan", "WingGeom", span)
        vsp.SetParmVal(wing_id, "TotalArea", "WingGeom", area)
        
        # 파일 저장
        vsp.Update()
        vsp3_save_path = os.path.join(OUTPUT_DIR, f"sample_{idx:03d}.vsp3")
        vsp.WriteVSPFile(vsp3_save_path)
        print(f"[OK] VSP3 저장 완료: {vsp3_save_path}")

        # DegenGeom 생성
        degen_geom_base = os.path.join(OUTPUT_DIR, f"sample_{idx:03d}_DegenGeom")
        vsp.SetComputationFileName(vsp.DEGEN_GEOM_CSV_TYPE, degen_geom_base + ".csv")
        vsp.SetComputationFileName(DEGEN_GEOM_VSPAERO_TYPE, degen_geom_base + ".vspaero")
        vsp.ComputeDegenGeom(vsp.SET_ALL, DEGEN_GEOM_VSPAERO_TYPE)
        print("[OK] DegenGeom 생성 완료")

        # vspaero 해석 준비
        vspaero_input = degen_geom_base + ".vspaero"
        if not os.path.exists(vspaero_input):
            raise FileNotFoundError(f"vspaero 입력 파일이 없습니다: {vspaero_input}")

        # vspaero 실행
        cmd = [
            VSPAERO_EXE_PATH,
            "-omp", "4",  # 병렬 4코어
            "-fs", "0.2", "END", "0", "10", "END", "0", "END",  # Mach 0.2, AoA 0~10, Beta 0
            vspaero_input
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)

        if result.returncode == 0:
            print(f"[OK] 해석 완료 (샘플 {idx})")
        else:
            print(f"[FAIL] 해석 실패 (샘플 {idx})")
            print("에러 내용:", result.stdout, result.stderr)

    except Exception as e:
        print(f"[ERROR] 샘플 {idx} 처리 중 에러 발생:", e)

# 샘플링 데이터 저장
sampling_data = pd.DataFrame({
    "Index": range(N_SAMPLES),
    "TotalSpan": spans,
    "TotalArea": areas
})
sampling_data.to_csv(os.path.join(OUTPUT_DIR, "sampling_data.csv"), index=False)

print("\n🎯 모든 작업 완료!")



🛠️ 샘플 0 처리 중...
[OK] VSP3 저장 완료: generated_vspaero_files\sample_000.vsp3
[OK] DegenGeom 생성 완료
[ERROR] 샘플 0 처리 중 에러 발생: vspaero 입력 파일이 없습니다: generated_vspaero_files\sample_000_DegenGeom.vspaero

🛠️ 샘플 1 처리 중...
[OK] VSP3 저장 완료: generated_vspaero_files\sample_001.vsp3
[OK] DegenGeom 생성 완료
[ERROR] 샘플 1 처리 중 에러 발생: vspaero 입력 파일이 없습니다: generated_vspaero_files\sample_001_DegenGeom.vspaero

🛠️ 샘플 2 처리 중...
[OK] VSP3 저장 완료: generated_vspaero_files\sample_002.vsp3
[OK] DegenGeom 생성 완료
[ERROR] 샘플 2 처리 중 에러 발생: vspaero 입력 파일이 없습니다: generated_vspaero_files\sample_002_DegenGeom.vspaero

🛠️ 샘플 3 처리 중...
[OK] VSP3 저장 완료: generated_vspaero_files\sample_003.vsp3
[OK] DegenGeom 생성 완료
[ERROR] 샘플 3 처리 중 에러 발생: vspaero 입력 파일이 없습니다: generated_vspaero_files\sample_003_DegenGeom.vspaero

🛠️ 샘플 4 처리 중...
[OK] VSP3 저장 완료: generated_vspaero_files\sample_004.vsp3
[OK] DegenGeom 생성 완료
[ERROR] 샘플 4 처리 중 에러 발생: vspaero 입력 파일이 없습니다: generated_vspaero_files\sample_004_DegenGeom.vspaero

🛠️ 샘플 5 처리 중...
[OK] VSP3 저장 